# FEVER on Google Colab
This version includes live stage logs and saved debug traces. Reuse the original `FEVER_Colab_Bundle.zip`; only the notebook needs updating.

Start with a GPU setup check. Then run **RoBERTa, Qwen or Llama** against the full **19,998-claim official FEVER development set**, with resumable outputs in Google Drive.

**This notebook contains no new results.** The included small training/validation sets only test the software. They are not final fine-tuning experiments. This benchmark has previously been inspected; do not call it an untouched test set.

1. Open this file in [Google Colab](https://colab.research.google.com/), then select **Runtime → Change runtime type → T4 GPU**.
2. In Google Drive, create a folder named **FEVER_Project** and upload both this notebook and `FEVER_Colab_Bundle.zip` into it. Leave the ZIP zipped. The notebook finds it automatically; you do not enter data or a file path.
3. Run sections 1–5. They verify the GPU, uploaded data and source hashes. They do not start training or the full evaluation.
4. Downloads and the one-claim Qwen check are already enabled. Section 5 prints the download estimate; section 6 runs the check. The first model download is approximately 5.75 GiB plus 0.08 GiB for embeddings.
5. Enable a long run only after inspecting that check. Free Colab sessions may end early. Drive checkpoints allow resuming, but free Colab does **not** guarantee background execution after closing your laptop.

Sources: [Colab limits](https://research.google.com/colaboratory/faq.html), [Qwen 3B](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct), [Llama 3.2 3B](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct).

## 1. Check your GPU
This prints the allocated GPU and available memory. It stops if the runtime is CPU-only.

In [ ]:
import sys, subprocess
from pathlib import Path
import torch
print("Python:", sys.version.split()[0])
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then run this cell again.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], check=True)

## 2. Install the Colab dependencies
This installs Python libraries in the **Colab runtime**, not on your laptop. It does not replace Colab's CUDA-enabled PyTorch. Model weights download later, after the size estimate.

In [ ]:
import subprocess, sys, os
os.environ["USE_TF"] = "0"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = "/content/fever_hf_cache"
packages = ["transformers==4.57.6", "accelerate>=1.10,<2", "bitsandbytes>=0.48,<0.49",
            "faiss-cpu==1.12.0", "scikit-learn>=1.6,<2", "requests>=2.32,<3", "librosa>=0.11,<0.12"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])
print("Libraries installed. If Colab requests a runtime restart, restart and rerun from section 1.")

## 3. Connect Drive — find the uploaded data automatically
Mounting Drive asks you to authenticate. The notebook never prints account tokens. Keep the bundle private; it does not need to be published anywhere.

Use **My Drive / FEVER_Project** for the two uploaded files. This cell creates that folder if necessary and finds the ZIP automatically. It also recognizes the earlier My Drive root location. The working extraction is temporary; new experiment results are saved under **FEVER_Project / runs**.

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile, json, sys

drive.mount("/content/drive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/FEVER_Project")
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
locations = [DRIVE_PROJECT / "FEVER_Colab_Bundle.zip",
             Path("/content/drive/MyDrive/FEVER_Colab_Bundle.zip")]
archive = next((p for p in locations if p.is_file()), None)
if archive is None:
    raise FileNotFoundError("The FEVER_Project folder is ready in My Drive. Upload FEVER_Colab_Bundle.zip into it, wait for the upload to finish, then rerun this cell. No path needs typing.")
print("Found your data:", archive)
PROJECT = Path("/content/fever_colab_project")
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    # Refuse traversal and links in an uploaded ZIP.
    for info in z.infolist():
        target = (PROJECT / info.filename).resolve()
        target.relative_to(PROJECT.resolve())
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Archive symlink rejected")
    z.extractall(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
import fever_colab_runtime as fever
EVAL_ROWS, EVAL_LABELS = fever.verify_bundle(PROJECT)
print("Project ready:", PROJECT)
# Install the debug runtime embedded in this notebook; the original Drive ZIP is unchanged.
import base64, zlib, hashlib, importlib
patch_source = zlib.decompress(base64.b64decode('eJy9fet220bS4H89BZw5OwBsCJJsx5uAQeYospzxN76tLGd2D8ODDyRBCREJYABQsuLROfs0+2D7JFu3vgAEKcUz++XMWATQl+rq6qrqqurqb7755qRcplMnnadVm9XOoqyd7HPetHlx4SzWy+V+Wrf5bJk5Z8c/h8670rkp6ysHSs3Lm2JZpvPGKQsnX1Vl3YZ7e++LzPn5wydnVc6zpZO2Tuq0+SobQYnVuk2n0FCV1fuzZZqvnNllNruqyrxoqZGXdX6dhc75Zd448L/UKbKbvb++Oni+P81bgAoqQlNFGzhFiQ3XWbVMb51y4bSXmZOl9TKHEbxfLtNV6nz3k7PK0mZdZ1gl3Pvmm2/2FnUJnZbLZTZr87JoBGznpFwXMHj+Pk/bDEFWH9VzgP/8XhbZnny4mAWXaXO5zKcBv4Ff4SprAZVtGvzWlEWwStvLoGyCOi3m5SqooZFsVS3yJbfGHVZQCKqq/j7A496b459O33yMx+7HTx8+vD87/+gG7tnpq0/np/jr3ftz5/Td+08//9V5/e7Ve3ey9/b9Syz/xf3HTVa4kfs/4M8B/vM0/Hb/2U/7r4umrdezFioTeqAIQrpPDwdv8N/9Z+HTTtG7vb15tnCayzRBiD2E04/2nMtYhh3Cp6ffvvD8Pecmby8Jci4VllVWeG49dX0nbZwF1CLKgkECrQCqPehxOk+jRVhn6dw7Onz6/DH+4wdT1/Wjy3BdId69KbRdZ+26LpzL8DL7PM8vsqaFHgmy/KJI4VvmldPf/EiV6wKH8xDO16uqwVJBAyhOrrLbJj6v11mQFUghSdrM8jx+lS6bzA+zYgbE6/n+RocIa1KXN43gQnocUx+0FrzGp4E2OFALH1SzzT5DS2FTLXOglSJrPN/JYRghIDyvPH9CnRRlvUqX+e8ZtKV6cB03/A3WideEs7TJFuVyrhoCOKXajafLK5oN8a2i23Ddzvwwb8oF9qDGdFPDfCRlMeMJDq7T5TrDaYYF82E9XebNJSyZVbXM2sz5j4/v38FKXd4CJ8iuYbHREpxlsBhVmblT1dk8pwV2MCsLHHSIi4/oPDY4GeG/YZXWuDxXV/O89vhBTQ1yoaS8oicgA8AU1aD3gDqkKXhn4Z4+W5j2H8U0mqhO8yZzfsHfp3Vd1p57qlicYUvI5xbpDBZ8vlhkdRM57hOYGAYW+hc63HNwCcfUF1J9UqSrzHND9wm9wqcnbgjkP4fmXbU0sI4sihtrTWji9AjQYDFEkKNFuFium0vPH5VNuGhui5kHr2BJFiXOvgNvZR487CcgiGlyF3WWASEhvWR1UNVlWwL7Q9TxK54O/g390N/754JmLubSB65qNcTBuF85T6qRgan6IJ/UvITOWdasgT8zYoHzl3V+kRfp0mmyFie1CVA+zS7LskHKfHf6d6deFw5OTYgzkgFWoz7da+RohmPYi/mGSAW6zxe3yXRdzIEr1mXZ4hjxL6OT3oxWsTVienfgrtIiXwA7YVR1uIK/R3wDYQyY5yAHWY1dnOfGnYQA7cpgUzNlahhrARK53gAKfyJQWdwC6pxFClXnSOBUE4G/aWLD3RhaFGMHGRBmkhfVum0I6qXrj+ps0QyMzpQHhQJwPDxKAB4I18N+AOSj77///jucLXz3pR67+dydECZqHD+WurOLfcEVKcX8XrlHMcy/h8D5gyu+AqkP7CldLp2j7wNqcLHIZzkQzhyY2bKsUFNwXr9sXIZT1AJqMqQF2hCxqvdfltEL+I/AWCIYLLXvhrr/VGQKAN0roQkYJ7SmukyLW8+l9y6NC8fsXsBKsx6z63wOMiqTVx0kDI4cQU+RHzuLjAi6cZAxp1CHZ+ogLUCjoiIECE0izlj8xUgjwDppbZuIB5YIvRbDBESfkipflq0mIGh+uDB8yOcER7fGHi7tJh7fB87nO/r9GX+PqWeUZ5MJ4RabGB9O/kx/jyaIS0R382c9XCO6scwQLt+p/ueiwpIMdkrgCcu0QkCrGtRZz/0FmUQOs81sIlIkl5nJoAaakcwBdAksF+T2yKGhM06Bm1lIARbYoBrQEhsTNoU4DJBEWV3DJZlcLpK2vIIlhfyirW9J/0Jd86IsL5ZZOCOdXxTOdZPViH0t4vSb8AIWlPvXV8n5+7+dviPK+DzLqtY5pT8AkVI4QARlxXVeA0/o12HFaVUCt1XKMayoqgSY/wErClZEUgNJN9Ba/A7UFJJOCOvl+gLY+gXI5Cy5XGtw/7o4rnIeVV4sypieQRuiHUeCr1Tz0uhmPwExVQ0OyzhCWNxD38CQUXgrleIMli5oV0IcJ7SMnHQ2y5oGmCuCAbqA8wqoilRskEtTWnayO+KCt+W6brIlLn7gTvO5o5CHhMi7s4/ZDBDd0P4rXbclqG/5zFnmM1AXMmqnalNgCaFDoDjtbZUhe8e/Hqi0CWkpSeIzZhHNsKSAjGMQB54HU5a2wFibwMWXbnDo4+o4tHRZDxEbNjkohCBg8et4ItprTaoICldQehqUyqAONekClMEC1G3gKHpZ0MS4ahrcgBu9TAP3JssvLlu9qXTSCoTu5xwGmi1v3aAGJjn3ELiDp48fPzsMnvqB+3P+kzMFZbjOHN4i1jwdzj/WKfz4ndaMtVK+uNi/G/WgiAwUBupEQZJMb1sQwBH2fUfELISOwoLWl9BlW9azS9RyZ1cpyOH4SxVtbgtDYBTYqVcxbiviVC7Vhb0ZrPmiQfUcSsEjTuwyQ6YADzDABiiEwQncArRGQIwLiuJV3u4vYfsLCHVBtDfN/qxau5M7a+AKKjdSvwL3AgpF1HM4W89psScgCIGoWKc9pPm1CuRNkl6D6oDqMmxdUI0iUgpc/KzakhFSFUYY6R0JPg/gK1/Qbn5rN4Mr7WOGW3j10vnROblMi4tMEwDSPbw9f462iACVROCTGSqBwNjXFa0jNG44Jx8+kW6YA/9tYQeQzUOLiUPlyA12oihwfzk7fusAKUaKSofLAzlXGWwxQIc49MMWxO0SGNCqrG+Foo98xSthLptsAFUXs1CMFx5rC9uRZn2BHUF7m8xSmAWotnf26d3567enydnpL68/vobtXOy4JA7259l0fbF//czd2wPRBIwJ0Pu2LHJoi7eDJ6AelGjAqcuLGjlXBdsSZ76uaf+Ew6MVh5pNeQGCraAtIvA6lISNtS9swr29c1Dcp0CLF4Q0mCJUCBjxNBcwmxmI1AZEKO1kndctzhJbfpgrXmVZtQ/C8Tpj846D6EtABORtknjIUgPZ+YC6M8+x54AQH+jNanwIPIgELPy6hDXUTkFFSpoMazTx08OguSxvEjVilhRr6CAW7YMVcpkmGgPwR3iFvYdDmyzrw0N2vdyQhj/Wv7ghIdqYUAQTsEhmopz6usBFlqhidh3V9uaoV+lnbwG8r/U2vvlBePitarqDmc6TantZzq5ijZbw7A08G8jKKgEKKVqrxOk1cVUugZRF5BKPJ/xmmtYxCy9uv4G+UFmE6cONAWEB/0kK1NRHAsNFQhtWC/MHnquqwia/2xJIbK11Si8tgLeKO63JZv6zG5C5CGCP3XW72P/OrgaMO/7imqZBinT7As6p5tONDJG6RKXIUJFYXU2tWEZ+Bq7QbSJl5TGA3u3/XIRiDXzfBTaH6iSKDKQJeEOEwK+ImAlCZugKxCK7SVhPdaNDKAaIQEVq3c7ciCxOgYurwY3w3zu1N7UoAZeH0kBFB23/MV+FqMXodQMv6LOeY3zj0cBixgGuadgyxWb886yZmdXAaxL/Cea3wJrzWQLTsmz0MlJq3GvqksQIiEJDqUyCFiWe0y8PUARMnOc+0cshmKfAugsxh1gN8OLyNA0QfRtik6XnBoYoUe2yCQuVJeJjIJ7pC3EYhcGuQHB2CwEt/WHLW85o4i7yKeg/fTHFcijRxTzf0rJgFrP6+r66qpRVFclhY78gOBfQeC3TcJv0mvgUEKsZNj/HXx4/NisKKI5sw10qVBgWmaH4lQZ4kz/u28wQR7rXXTXANr+iMcNsCXkwiSLlZe3TtBJqyIzY40k03EGWZBkU2ZjI9isyq1gmbm4Blssc+erTIXPiJsMa9Q2INlQClBhz9GTdsnzF/pVNCkXzpmCIlNnUQZN37I7dJzRl46Oj6Oj7yRPX+XR+MnFkzHpBw/t/4gYG2heuIrxBKQFIOpF6ydjwsAdEEBn4WIfDVwGZTxUrIBWeliWNgOYsePwYVJQlyDccCxkWtfyKZJ1ZoiCcLUugCzM0+o4NPYoNofJOmJmtHxkG1xXJA8RkNxiLoqFZhzBS2DtcZbe8ezCKlYtioVq3vH3FbUK5bjvPdYpTAyppdgNPmVqX7iSyAK/KyoPWA9qO7zm2PBvLgCYx/R1ZtcRlI4jEajWsm3oOInBopd4nDrv0YIuoPor1F2idZhaKdGb1zhoDzR+Ti7VuGNKh5fLE/RXofmTXFmu8brTDu/g1rID4y1V0zRMVXONEMSzKkIszfEWUnCOLZm1y+/zc3dnd3XoyaudP7KSwJ0ijY+LzKrKodwBn9oLiPZ3r+k8IgVDdfWJzFxjWEI4UxcoITQdA9wZk9wRfoVmCy2kD3jg6+u5wMtSIjY1OU2fpjcOk7ch3u1272mC7Bq+dVlktsBqy1odwDpzqufC+GhgmiylRnOnN2L0BQaplxqSny4ZphbzcUwr1Xnd1GV1v8iQ+GnUW3qZOhmV6DXTVQiiAXFAAY8PyBHjK63e/HL95/ZLEySpLC7IDdQEd7397GE38A7TMD34Z1dmKTZSxDYB0vL9lVMie2zTWdR9j98or0O3H/zF+ZqwMDxEDwoKOfL1dAO7SJlXZtIv8s4eMMu7iQm//duAQrasgyi6F2nu6nbBnpAs3MKy434/Zcm5BDG9Nh1BJmkkHpp3QCkBCk2I54yfUSYDVAdwJor1XCF8pRQjeXFwmepaSSzRSSjmYv4NnLw5Rx8MpgcdNHkLbCIbbyB/GCS/Bjpzyg46csst1BZgAZ9a3XRTmqAKyQWOZ6zNTCfQKtgtmuM6lLTNjRiG3ivY+U9NaRdf7AaOx3lzmy8xShfQmN7xJcy63ueNm7WBA7bhX73D+5Pwtyypyg6YA5nXGasPISR3djZodbbchz0cJdWqxRM+AY4XUnqi88R/SdI98W7sZlrBdJUDEtKthdO9TDB4i/fds7X2zpKhiwTbtXl7c7ZD5oy0C35LM7sc2Xy4pTAoDDP7pbBHUfehFWD8ZhhrpThWAZgRUfEXm+fYyF6uZ0tGJVLSKi7sot0rXDTIpGFBjOVs2KBXdqN1dLcWeIDHAaoyf+l+nI/eUSLRKkBYJP0ZDm2WQkgitQC87G4Gd//h9vY4H7dvQbJUVqqxEz7T54tZTK2RgV4BhDWr92O2pl16vCrebFw16fVk8eIA9yxSppp4mgu2MIGiyGn27LGoCCiown9FkLUKILbJnaDOltkHXbtbLtnHSBYbx5YU0NHJ0m8rRyGbZDJezLH/ixRykY5ssAVz/wOKAjXugoedwniYeV8aPIVbNi2U59dzHsmMkxFFYzKO4FyvCblmFBsQqYdTsJqkL2ci570rbgiyjvs3a0DlB/wIMhUzVuOaYVhwYbXkTUsCCZW3gOCS0cVLzAWx0Yo5Gc6qoIgLF2CqQkGJJVBEjdtSLh4tQQe6LyRCXXhvvCnTZc0DPWqX1LW4OWlrfV+wIuuKtHDCCQO8FWHUIXPZsqieYgiZvWuSfgY4FEGZ0//6vrdcF2XjcvqVw4z8XWUIFDJKd7uiG66q2gXuRFWL0t16ydEX3kzIGahKUuUWy1tM9GPGDwyKhBhTzG7p6qAqaDHlBbVA1m10ZubABUF8TpfLa4SomSudhQStj0gkk6GRi97NK29ll1pgBIkdTH7W2vR0s5Wmy91hceKcJxx/JmvigF4SDTtjIldgvR9OFCbrj1ayDRw6U7GGBooZnkbKaPeQ6OHHS56/F6f88Pjl3Ppy9f/vhHATlk1apSqDMIlGelT+dnp1jfC46u2GVEk2SP1opavyBA4swoBX1JdKgSbkyXb369OaNc3b8d+fs9OOH9+8+nlr9GXXPQp+BZZpO82WODjcQ+Qqb3DwqrTaCOrFq+pNNM93iNnGMNKwYqvvm9elL58Pxx4/HP59+RFgt0FQbItbXVbXMAQd6Rvzdcy5CrdcKL3sfeOvZ6fHL/+VGZpZ+ef3y9N3JqfPq+PWbT2enzsvTc/i1Dar7OldBcUyeLN2E5SpNqCvelinwUBLpTfxdP+SQiVHqW2KlG1pomx73bKFgJswIB5v/swYBPaJrEdira+DqSQPRLnYx7dHmGr3fygoYQydojPHGIPk7olEpOI+Vo+lBMgi3yNCiDuUZiGOfY6SLUs6w8Hj/aMKeKok5hXW3jJeAO4/KeosABOES417MbHVX4Nnpyem7c+f0F/iXaMeVUGRsypD/K/SmQ5cR+TaoY4toCF972q+cNUhz7elqms0BLUqXAe3UeQvbzTdvHQ4HAW1mVpdQQ4VK6UjP2eW6IPUaY2auAQ0YpwJ7idlluMUTrCIwOg5bcaOwW8qKvVAFjtdteY6iM/89qwN8eouyWDuNsH5M/4q6rMrGnZohdoA7VorpynCnzcCMOSJlYoKV7C8SnzIJQGjD7EgQFQanG0vEn5yPKJsdDlqvEdW3dJjiBAMf0EOOWGJ/OUUqYEgwaRwoXWk+0Dtfh2pMpGzEeqj/v0APUeZ6GAgBcqhas1+DvEposCnWq4S3Ho33VNnsOSxfux2aQPue9U5Gpo1iY5Dgi4p8TagZx+OJHEKgXSsFL2LMiHcYoOWJWvSDZ09ZBkxRsMf0ckzlI/r3ybOnE+HDGDqoTVZ63tXhgc9BOoedJagqOcYWslWDHBD+j0+/fWGiFKmjwahNWhw4eRSZiHaMbN4gqf6eFQ60sU+tOst8lbcuuzipbzEwaZA86gIUA2pNYt1YC8RZk40GLdREIrZAUZcmzTaPJ0fvK5IVnZFgK8VlDsu4iA39eI8fCzB+SJyFS5CgyEartLmK5fvYTVvUZZEf4nt3Eq6LBlgTxq3vi2HhOva4gcdYBBjjeuUd+Qf4IL9hN5euKm+VF7HU4Vk31lY9hKIIFzB6jlQJTUDpdVABK5/nK2ghJAoC8gTYMYTOpXCIZ09dLYaJHOL40FEEBaTxY2woCd979OHg4NnTJ0f+fzs6hOJRf6PpZmqeE5onN+AXMI0rPO8hTVs06rOhkKRTY/VojigA2aPjBNX8As2hjArZkcqRjERtxUgRDjQUA6HsVhiZ1lUil8KnMPbG+Xt+lcM2IU8dOSoGjHqR100L/8LncmEOlMGGYr/NaOVdgV746ewNlEX2JTUPhJWj8aSp0lnWkLOzrEHlwA2LwZYb6Z+wJUJpkLCrXm11oqeHh4HL7SUzEBMYQ4sBcM8OaQ9FSJhdpnU6g41wMl3PobIb/Xe0a0JHvI3iM0Ru1A233xoZH7hNua5xNyb1vhT9qk09cw8K3vMVvOdb4H48oRisRALYQgrtW2RAEMmNQm+i0Mtfs+oyA1kFQNTpBb6a3N11Zhi6wDB7MiKgfVVRVm1me5OFkijc1rHir3rGj/nDKyyf1VK7A5iqIk0kNFVNQK6gX2hyXgIypynwIIxMTAT4zfMUaHnSquOIqDe+j5ZHcv4Fqlg7Hz7qgRoVzTGpInl7G5vzHl9wLxQB7nhTpHbk/EacVuiqQyLWyKHPym3V/RKMJ0AbCkqMJe2cLPHvfDkd1QW0t0XT8PQ2aRs7GNpJ3WOHIB6G5cZuFw3u5FHcfTMgnE4wmnCu9yKMyQM1QGeVN7QzJhmyyfHUXopiEkE0tLhtbRqKmiWRinBt2yUFX+6ABfbsHrEMRZkVLV5IX/ZUdOdgoBzTJAWZ0fKln2TFoF/zKa2OkWZKEoJGXoL4YbTSw3HUfQ5MlNbp2dn7M/dOR+3r02PIQ8JzxQxf5jUtHsRrtsg/x8xF+FDCPp8wQ3jZsaABpwWEj7A0eM3GW9ayx5Mzz+tYVz9wNUX73bAudCjIAsdN0WeMIOHDLsDhMhCmFMTs7Vwg4+jbSXAkGsUm1RC8KCalkhvIj7jbMwtH1TIJ0W8DOXG0vXMgqXVNbjUCe+zCk7iOHT20e0I2LIToOrGgOaS/Xr/9scIo+UXJj8AUqPSWL1eRlBhfTWxDIZAN4oDaAdaC/gyyHSrVu/fExfQjjgBGs6roZVvn2bXEUwWuOgephNjkTkbHm64GWE2LoHU5unrUQsYooL6qvzGnqglGEgBMo1ITqgY5CfQEdNyVQ86p3lShN5PhS5oyWaQ18Rd+o8Gioxhcj98Mnvj4PFNTy5xBzxBNQdSbWWUFjfgExueZOYPxxFUnS/E1UP3hoUHxbhQlfHDPHaJVy9PJAOIWfKyMsdT+ArfOy9tI0aTlLRFnNfeiQxeGKH3fhA8riwzjc/C0VDeFgLZdYfw9Te4+HiZGXRnPuO1rebIAODE4PGRGs0Np1hGVwsTt+R1Vl6BXbFuurBo2sSZY2bqNP4+t9lFgyolE2q9x25MnY5u/T4wepX74I0GpBepOtBKsg4MVHZbNG2bAGw2rZbG1X1gNXb4Nkm1AEfMshj+Xdwq/gWBtHKGBZ1Slt6hhBLNcTg/GthLndWTgfBryEDzVxv5RNAmeHcpCVMSkmhqmJ6UQOEpjAKpuyZYBUhBeS9eO6PPcNPvzJThEeUPZXhrIEGI1lA2NI9YQDdCYYU2iOcjfgFeSKBH8p7MG1ZDn0443EZAkq1KpHVsOoAE76oyKuojv5zZHh8BuhqDbNhI8mGxg7vQJqgcdWbTcP7s4huqhSWA3lggRJMB9KSpGE51yDCFXQLzoD1rHDaww9JgCCnxzjoipf1dX7mTwTJHWrfqbUfiZFutKTk3Tkf7e2XGOBNvboepKeFBfbRVoldpqiK+jD1vhKNv04aGp6LCDgQLEEHYGyJitP31jAwI006AfpIapwKUWyJIL+u5C8ZpTLEzv08Dafs/RfDCztLXIW9j6GwOXOueXSnoMahGn4k/OMZ1+pKwUTlo4ZcWWncBkrXib1ld4nA/mDYV2HSgX+GccVktZLbAl2r0265oQh+YKmMiG6BCRl6NXXbV5QGauffF/Kr6DeWacaTm/JcuhSusBCwhnah7XWYjikLZGXu3+53/+p/eXCLdp/l/Gzq/t5PGv9V9+Lbzw8V98+gUF3ACbC6Dmy/fnx2/e/BN+vf753fuz05Pjj6dM9tx6RP3y7xAPVVUeWsIqex+IRcxSAVWwaOjYqFcF6EYkoxUGf1S+Ckz18OggzLR7F6hfyi99xzYu67R6Rdb4Sns+VSN8LN5XC9rudyyNTgJ0Sugi5v0ApbzmeDcyJi3x4L5DZZ0GCGeVupJJAcNsCSjTlAkssEDAct3R4xv/UczjdVH+1vjd9rbn5Oj+bd3Aes/ZkOre7QAVW7LgU/bjAVBAzYeFVBtk8TtFS/YWQMDrQjGhmlRLA96biPvgJPkJjajqBDDwE260g4PJMErnjZnOoXHqEeYKMuEh4wEGx8pXTpmD5mxYZbt7ix1hPgZ8xJ87BqY1S8nugFD3kPQo3sirRPRMEM6H1JGPACu59w5gD76m1DeIv4IM9anWZLA/6aWJx7k9L3pkhkon6O3ARFjzWGd8Qmith0I3J/HaOkkUEdZy6X2OY+ulVl5VNTm7uzFey9LLSzgSULoxKFHFUoKf0LJFoEfWajaxJXa8imouji0mgWOzeUg3fFQaoQKimriRK+0k5JyS/AgkNZlBWcAmq7xpyF6sOhGKq+zmVShBwqmRMDuWyIyEDG2GzQrm6O2dcmiepOsmXUrKB3RnDrsfA5Ur5l/1P74qa+7yzdvgJ5CUx8X8JzwWflIWixxPn9onrkfDrkoEiFx2/MjBG3EnNobQmfCXsIV9UCcsxjhBH+7zHHYadh2GQ2kYdnlAew64CgGkNvqeubLhD1CHEgTEm6jzqN+8SJ5P85ZdZdNiSk8J1UlQw47dYvHcNV/WQDjzcj0F5Zkb7lZEnQG4QzKnuuyEIn/S0QsNf8/rak3wfwkeAztjAm7fABcxvQvk+PoqrUAiutHhHZ+HHxyNOHWZ/iVYj8kf/RBKT1yln+kMg3hFvz16Sq/kc/z88PsXQ67dhSUkRF82US9DWW+E41uar6jwIzlF/u70tRVS6HLYAy0Da0lgLPCvxet3Hz6dk6IY9eJnNFCygURLzGY4CB2zEJV+/MWtyyWyMcyz4oo3Ctkj93g3Iet1AUxuw5mLVqflLfqtgBRhw7LEfZhqOVDFZFLRBW1F6MnY1InYLd5i1fGQA5ubfZDzmB37mBYC+6K9IJ1E0W5fZQHH3FbNZVplY4oY2dxDMdgJcp5bzBhiHROgZgN9QLV7hqBLZor6EtpOxBbB+Yq4oK0n3Uo/2uU4Noy22RihCDD1nfJd2wNSmdNgahTclSqUuYMmHj3G38ik5/aNeXKariub9SEeVV8RENp0eRIT9UHPajc+NGIM9qJE8aQ3i1A8F4BnDnmelU2T/twNzZWituIiMdF5m1Pmj+zA0Z1W9QcFIDD4VvxBqDZmJhChz3Z65DEvYY+OezuhaS1GknxjDdrfEEa9C4wZkPFhQEOIJqM6venXnmdkZtSVguYqrwZDRXCRls3AsAyTNlINYBlhafg/KbdG3YY3oo6T7jKGZwpihf1TPC3LJYWxaHB8UnpxqepXFMxFnpyyGVyiWn3Cs0U5WTq6a7Hbw4alIMYXvgnVGSLxgVDjaMjYtElZf2h5aLIFkXLzsNXSG9xm2HSEb+jId30bSVyKMqFZxhTs0LalkF1UjH+eEW3BOQhe/vW37JZ++MokyI1rq3+s7H47J03rz6TmqiZEPX9kzgqK6qvrybmzzqG3Xm2Lj8m4e+YlncSMvvZPrsjhTFaOyAfb0zjpnU6Yo+MLy2lWt+lOfZzsdsgDYPYu2kvUQv5NmvlHDKyEdXcisXXM9x+olBuAzE9R0LvZU/+o9k3jxfQOy4STn6G9TAeD7dStN3XT4SH+O/q01AatTOqjPMwDhfFhiCCH7D+Kn+3K94hxhNk+51pUkekm8bR7n7ba0USDx4/XBR6i+i9USf/kfBANx0p0qrwgZT0ne2YpBlBt55jlOPbpLbTJaEP2g+q1TiWM/lXj3Ko6Vhity+JPVi6lXCbWCmOeQRhBu1Hy19Ikbe8P4bKvOfodNXCHTrqrJRcHLjLBtZd0b0U9QE3d5JHMSRKtdfS0mfs12gAkhjpyY0tDwpgf6G/Wyx97gNMS0GrPv6gu4cGIbYGa5QVsiMOmXLR4Omr/yB8fTnhv5/nhDPOWQKsUPG5xbi2r2bg4xqocWPusE9eeU9fjfOJTOIwezi5ZvjHqoHewg9rsS+mdE8JWH3dzVliOy7RYx6Si7RMT9A+yRMOq3hBAiNouSjEpzL8o/nSyPWJtOryom638C5/GM4GJdipCOa5gZS7HbO2YpcyXk3x5w7VYT6zCZr1Y5J85iQVHpQUuGYuCbhJIeJzmlMeCktytSYPlIxkepzBVYYqUA/kqL+Y6Zt8KWFzlxbqFTfTzb7fZEYxAHzrhNpDObTDtm2qcTx4w85fNIJRa40lrpQDFR4dyHPO4xT15i+ll6lsOikPECJPmtUmZ7fCIBOEP9Z4Vpc9TZ1Zof9rwKYaNTIn9zMkDQgYPYuxjSkMOh3VWIGgdPsc/mPpYHa3AwxfoRldpaJ1mBXhlcuIAL85LK0GHW3Dxw9F2I/y+JOxA3ZakMszFcs4ATkEolk2OTakOeKJ/iA+HLO+CLCm0tY3OJHaSFCDy+36JbnHg9OzB6L7+4WgwvTNlGlfd48UVkh+YppPykWO3lIyU7oIImwzW2fOn/ogZ9Sot1ikerey85Qxb5hOm6MLP94XHylErCiftnN/FtQWoUSGvaoFRMhpYcaC9iMgTbX+Im5hzHpQRYUL5MjnyE3Zu2Iwb0fJ1rXhHE+mYIVN9/jT4yhDrPUdZS7bU3LRaB66EVW+r0tSzgy1R2L7l9TMBvPcFHiOUXfbkRj2Dg2uZlgAfaPTU6oZhY1xNiRuSJ+zMAP6VzW9tGxPq3RRKRZc0sMeaY+ZNvloUjlbyWg1EfxlztwMfqD2dwjppZmWFhsyhROqBsBvgNqLdFqAntuWa/PstZriXU0jIBV4cyoIx17xgFOwCRpvVdMYs3nKZwUhSo1H8LkU+j/W5WpeOhqMrXx9BqLpHZhsVWUCcGAr83//9f9xuwjDKLl7eqHTnkTpCbh8FH4zNHgzOvveEOCf1kyQpczzaqRsnhHXSK8lH0Y2t7wo9JuYYylrYHGKqdKLews+8zJhjUsSCsnLqVcDZFyhWW2IhW5N8jedEfQAgTdyBfJOjmyTt3XRJ9l0ds6EP7e+pc52xyUbL8fz68Ci1QP9o8Whl29EB5b5J9bNeee1QNiQ+F0d6ENcYPCK/qS9svAl25EbF09kSchhfi4lCtLsdMerAOeLDkb0iJXrrcKTy/1AOjKv4UAK2VIyRSvsxktQZ6vipmKlVxI2rQ84FsSodBxIqGnRNQCEppLDVIr1Uv1Y+KaIBNcD+UU/P8MjtPZlTiaoTnT5bDn5DP0H3HYkiH7MbqZxUOPFCarSsNK575qENeWZpkb4SjHFfMHZdvrqNQE5M9YdGw2iUE4OSUD8gmtlEMftydlH4kFpCiltsr/tjLDrS4xeH0RQJROrcoxvBpx/jbplIEZBOqE9XeSlysvQdTVMj3eMfOSwx6N/ocbPI4mVDeWnvdgf/9qanm2LRZGYxkNKrDrMNWNsDhBgW8wSw9eSoG2Sr/JpfdSyrA6cVzNqJw1EHd6wTTMq3NeqX1ol3NixXm5GyasWEyji2w4vb0262DMAf5GB7JubeUnvfr9v3i7eUO5Xz8hGAvYnD/KqAw6RcqDSrlsnZpZvkmC6ZLzRq5ZDiZIm6Gzwdn07rnDb9EnbuDIWJ4uV20CMea+QeQ+cVHqmmS+/oxQi05rTAqyvY9HagriVycOHkeP0KXW2Hm2Doyb6SYdDeb/evTf7GC+B3jieo+RYHwAMDggemhTMLblkp6hwC/004U6kq2c0T2g8w7R5JGMyYuDs+XqpAGZQVwLQw6QrsMWfL9RxRy1nQrGwQeDnLJSZKIoOFZXq9wWsBYCKWtyAL82ULE0dsy5xD4E4UcwTS2W/L/QybyasMU8s66IAvZreha4HWzxnHfph71EU+bm0Ci+8pLjHHqBdgnkiNe8ozpeKRTdZA0Q26j0+OSHhsyw3Jgu5Q5MUmhfyo7e69yxnY/yWZqJ7ZNdGOWad8vyCwqYO+IRzvJgqd15LjB/cGyxLWKud64l6VLUNO4bhKY+4OTXvu+xuYB0DMq6PbHiZas8chOBLPswUz7zubA+Ul0/sL5JsgiYn3DN2VB5Rbk2eAgxlCtYHoCpc41jpu1NXytOas9TzL2rOROMvVxwqAoUzLtJ6/xnVWryuKc+g2natPthaJsSuO9QXaI+SUBYxvXeOVBrDd45kbZMHuppZkd9RxHVKytxFN3fbzEBtwC5dSIN/HCzfhQSvl17G6HsidExSsF/S7Iiuu1q4bTddKCHeOiWjJbB3h0jp3zypszpyoLrmSjatA59OTfbMysMm5BaC8wEWRxZpg6HykahGgxW7Fxbv50Pa4LkgEwtRr2deWvMXOwu72mm/ynMES8ToW4CYQaclZ76iwuRGlk+GD4/av6AocjGsEKa79tOkMiDGd3SbUBbqqZ3wEEn6g0FnQ+6ThUF7UbhbrhpNRQDuf7zO0sXHq4Vfn4TG3r7w6b6SCIzixmo0dkmwUBU297bIOs/twxpJR6yPIaZlViBVGG3TYiI2WEhVYj5ODexCaJL7YdYexUUwyXzCpUbaKbNNHZ3Rfn0WQrxlgZ2+85X5Axou6nwL2i8xR91W93Rf2WXqiFT6+MQGm3UexbhhXpm3oeP3u5P3bD29Oz08xU5KxTbgHbCrjq4lolhTpokyZr2ckE0YiRlATIrdCPK77w+T0JBYalD1jMrqNx0iBY/uzdd0itTgZUZrkMR+p75Q0Qdu9KuYYAD1HFpR6xQW84oLFUZDE96xD75aCSgJeDTG7FoPfs7pM5rlEvB4Sw6ZVGveX7WD9J2Ot2uBp82dBNOn6Myl0Badpgre2qBP5WrAqaUxD9GO5lJKDYtDojOFbgIKEZg9v6GLfIReny7xoOt2oy5MEVjLGzmCAiyPY3JLTtahCTMnsLY58NkW52jyLNUJSjj2TyzuwNU9YH+sV2p03B08Rpx18ob0X/3IvFRrC0QbjRl8Ig9EXV0+YAk6/QJcuekJxGtVHfuIvZjyLI3zDVzTmPDfd/AByUOiOwWBOzhp+5L7hyzI7ESdyFfErlYpmfyMVjTlO6sxLItBbTM9KB8UUi8OjNaC5vTr95fSMJVFITMVatnYe46vIppTesXydhrNwrRm3plbf24Pn6v09E3EVUajHWYbkz7qgHHJgM/UIZGr+j3UGexXsF08/aWWRU+CX68YRMjzQR99qbi8k+safxKY9wW2jkj92703Sm4yRtR/h6oHkElJp3xhevPqMfpgDgxLQRdJ9XbDpPqFUUMv0Nqsbj9ScoLAOCR79EBc/kH7LESticJNz2XVIFXcd8qECHMirhBVJFG4ONxowbOzbj1D4sOxLLup0zuEoG1Ws9Gg7a6NywpWnmA3ZNDA4hPF+EU3UjejUG9V6SBdWzAVgkqrBIuXLMQRSTEoODII82y5FYqF7OTGNuxHavysMoMqWnr8LTSx1O4D4cjnV1zXoy53ywCJhJEWWCIKM6meRSHwUkH8JBD45nrXv/5nyt5+ImJlTg/vQImo1HLMHGg1bxoATGe+P8Wc6aKJkyylfRpezBWHufP/i4PsXDrlFKaaK73vW97E6GCBzAwtUGA/B2FGl9tQZVMdah5JkwzQDKgIuKDmLqrjR/jy71mXkWjs7DCDY1Hz/HeGJwTnF7NX8F7o+ri/W6IRr1JcTSckQYJqAk3K5TAGev+ft5QdOLIdGbfJRD0AVckRgncD+cKlhxGRdNN1mUnbr8osjlph7/VDK1vjNd6rrnNOW5ktfvCK0RfnMBs9wnvA94nwpL5A2FfTlzmNyBJIO72277lj3GStCFp3BKo9tmlvngJtttLvlZuTdjVuVuj2Yw/I8BO2hVMdOBQTjuWS8vAJR9K5sX6HHwoo3MURNqwaFl+CXIkrIrF7LTciwd8wyB2Mk356S2qwXy/6+LMH93hoJnZcsu5v1FJZxC0ygc3V3qKfD2ojpsfWvm5ahAWlIEw/fvcH+nljEvddiq6Ypj9ODLtImcO9G1w8qfE0tk+0Oy//5Wh3Z9fjFP+mF/2cF7wBVq3Vu0ciBdTc1N2hutbbIjOhf7zgJbv+H+PsXQ4m8ac0QTcz4buRp5vCZEr5+VU20uXOVegnXeEzf80WAoaeLN0vcW+AaoOUDvIDXfwViLWvi+IbhNpjpvsCcHERDaMXoJKU5O/7Z0K94N9iuqC4dZKowXOpAxWnoINhOwI0eFKxdVRL0LBzYE5c4HmtdlqjzVcjOQ64EfWjIt4A/HIHt7zlfHdd9X8Na64h3aH726EWb1PW6EUUSZEJ/hCIS2Vtt0oP6wgTBhTeifgxv6FTdKKdYBe5H/rUYIkGZJercaCi2Ssr5dpwUaNazS0xoSWCQZKQDVngtdPQ02/+Ww3hUwBAGPrqUSzShy7yj54GLmluOt/7glmS1XvJRNPzCiZaTOWzYYGcaHh7hhZ40aRRnpKck2BFGhHE6w7E5xDiUK0PvpJV1q5/E+1iCQITO0LooNXlLrVN3d1/vOXIKG/kkaidoiGFHIukbIUUJyAeKwdk8DsIWdQqvxZ/o6G10QcCslNNBtDrzL1XUJUGlwawCqtWcI37qWBeFDfAIS8RWcLvxKNeWo5Zj8x8U6I5nV8haTAkN5D6EiVg9QkqLh72I7UZjEYsPhtco9KoIm+DBbEzusOrfmsxH6vjaZBT3Kxw46AUZyRZtTN8IzcUdsaif6sgD2yo1v56m9Ug1oC6PVhdSk1X+YIn5wzmxeagoBf09x+1PFP/i9fTbYQLx9WVCO4LvsRYm1yOmM80ucqaRIK0v6EaZNiPveV0ug8ePr27wtT/kC1CjswISzOVhfDHQspyS2T2r1LVh9B6JAl82AQNBtGwxSovZ6U/EKYc2irF+OR7eSE6sQWOvf2jM+cIZ9qI8io0EjbZg5isw0odVZYW+H1KK21aA0G1HHcBVtMQfnyNi7PKWflvJpbsNcPJoyTXd/RT3m92GM/r6NaS0Fcz7LtX90Qo9Gool0JBxJFqiooZ2wejrmBqYKGQZ6+WcsmsRwxj1PlhjLyuxoeiTK1LW0MWyvNhKEnSZgxwBe8jSla3rbnxLoZiOA8DkfrmzqNSy5WLhf8OyUqt/YGFZn+7DvoAnm4VsK1xqcPcjzeJK2tj+ILwp+0AfdUgP/yLrtZQz8YHuAsiUtkM9rC1DQ8mc9eO+XAa3gV+6nwahizcMMZ4cO8aUvNs68QO6OYF4P6uNMaaOtzTGmBRGW+uLUelDczeorpLsQySY1iHj54H1lXbG1sfvgkH1krkHVF1URy94J4OeumsaKoJycRuzbusGcp/e0Fu2NnLWhqe4DC8UL2viI1A0QE2OQUtGHS9RD2yMhqrxmA58URaUaYb3qVBag7RF9s8wiZcB9tZWidgy2F9QbGGd5A0UaOGXHosetpkCfcPC8LeESQ9vRKbT8lCgTlGfJg3pjkaBwEJ3OJGY+hAF8aEcUpVMMl5l3e1uOY3EPOZVIac+wjxcVWi5ZEPoXI72KdfY+DA4Cp5OAkB1jZEN3J674Wu7k20dDZ/+ehKaS7s5GhUtNqadOWvbcVcnZyWESGCwAKkioAHOMsrZxE4ooyojzmmeZ2J+jLfYIj1ThXM1JECUbQ7LNykX8XccHo4peRQjkedgJkog4KWjI/p8yIW9LPGA4dLbuiL97hbI/mSsbtYlt7Sn0fHggwHcxhJ9TyS3zBibXz0Ov+CcWRbDUsMSt7aqs8Gbfoy7n4zCsafymsbdjdmox1xhPeOscsiIBJHrO1fprU8Bq9wJLn4+o8qfRuZcJX2zDBJcYDiArrfrDPr2BBqCbHSHhg17aWQLtoyy9vDdahaLMYUCxEbeXHbDk/nwxM7YLT7ovq5tpwFBa+LHXOWHU9JCf1KhOd2jLHI4oxvj06UeFW1nBqCCekgP/gORPTsjzP5AbBmFlHEP7B7AmMumtaLojPnOTM0DosW+Mk6sT9ZaZ1ChYg8JA+vGYnWjsEaYkVwoiw9A2OFcygaCbe39P0PjZLk='))
expected_runtime_sha256 = '6d0c28025477ec6b454c25f69f0cec31dee278a8b4a7bfe9be6301350a971237'
assert hashlib.sha256(patch_source).hexdigest() == expected_runtime_sha256
manifest_path = PROJECT / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
if manifest['files']['src/fever_colab_runtime.py'] not in ['f2ca840db6e6d871bb42a1bd3ab84ef77613d7946541add86c407b7c4b26ad55', '6d0c28025477ec6b454c25f69f0cec31dee278a8b4a7bfe9be6301350a971237']:
    raise RuntimeError('Unexpected bundle revision; use the original supplied ZIP.')
(PROJECT / 'src/fever_colab_runtime.py').write_bytes(patch_source)
manifest['files']['src/fever_colab_runtime.py'] = expected_runtime_sha256
manifest['notebook_patch'] = {'revision': 'colab-debug-v3', 'runtime_sha256': expected_runtime_sha256}
manifest_path.write_text(json.dumps(manifest, indent=2))
importlib.invalidate_caches()
fever = importlib.reload(fever)
assert fever.RUNTIME_REVISION == 'colab-debug-v3'
EVAL_ROWS, EVAL_LABELS = fever.verify_bundle(PROJECT)
print('Debug runtime ready. The original ZIP and earlier experiment results were preserved.')


## 4. Experiment settings
These settings are recorded and checked on resume. Changing the model, prompt, token budget, source code or package/GPU identity requires a **new experiment name**.

- **Qwen:** Qwen2.5-3B-Instruct in 4-bit NF4, the first setup test.
- **Llama:** Llama-3.2-3B-Instruct in 4-bit NF4. You must obtain access on its model page yourself and add `HF_TOKEN` in Colab Secrets. This is a new 3B model, **not** the earlier Ollama Llama 3.1 8B.
- **RoBERTa:** the exact existing project checkpoint, with 512-token inputs.
- Only one classifier is loaded at a time. RAG contexts are shared between conditions; different tokenizers and RoBERTa's shorter context remain limitations.
- This HF generator uses ordinary greedy decoding, not Ollama's JSON-schema constrained decoder. The parser accepts plain JSON or one complete Markdown JSON fence. It derives the overall label from validated parts, records any missing model label, and preserves the exact response. Malformed, truncated or invalid-citation outputs remain errors; no content is repaired. The 512-token output cap is a declared new setting.

In [ ]:
EXPERIMENT_NAME = "colab_full_dev_debug_v3" # @param {type:"string"}
MODEL_FAMILY = "qwen" # @param ["qwen", "llama", "roberta"]
ALLOW_MODEL_DOWNLOADS = True # @param {type:"boolean"}
SESSION_MINUTES = 45 # @param {type:"integer"}
MAX_NEW_TOKENS = 512 # @param {type:"integer"}
ROBERTA_MAX_LENGTH = 512 # @param [256, 512] {type:"raw"}
# Leave empty for the included original checkpoint. A completed fine-tuned model may be selected later.
ROBERTA_CHECKPOINT_OVERRIDE = "" # @param {type:"string"}
RUN_ROOT = DRIVE_PROJECT / "runs" / EXPERIMENT_NAME
previous_run = Path("/content/drive/MyDrive/FEVER_runs") / EXPERIMENT_NAME
if previous_run.is_dir() and not RUN_ROOT.exists():
    RUN_ROOT = previous_run  # Resume an existing run without moving or replacing it.
RUN_ROOT.mkdir(parents=True, exist_ok=True)
if not 1 <= SESSION_MINUTES <= 300:
    raise ValueError("Use a bounded session between 1 and 300 minutes.")
if not 1 <= MAX_NEW_TOKENS <= 1024:
    raise ValueError("Use 1–1024 output tokens; freeze the choice before the full evaluation.")
if ROBERTA_MAX_LENGTH not in (256, 512):
    raise ValueError("RoBERTa supports 256 or 512 here, not arbitrarily long article input.")
print("Results saved to:", RUN_ROOT)

## 5. Check model access and report download sizes
This reads model metadata; it does not download model weights. The listed weight sizes are disk/network downloads **before** runtime quantization. Four-bit loading lowers GPU memory requirements, not necessarily network transfer.

If Llama access fails, use Qwen for the setup check while requesting Llama access separately. No token is written to the experiment protocol.

In [ ]:
EMBEDDING = fever.remote_metadata("sentence-transformers/all-MiniLM-L6-v2")
if MODEL_FAMILY == "roberta":
    MODEL_METADATA = {"path": ROBERTA_CHECKPOINT_OVERRIDE or str(PROJECT / "checkpoint/roberta_evidence")}
    CONDITION = "roberta_" + ("selected_checkpoint" if ROBERTA_CHECKPOINT_OVERRIDE else "existing") + "_rag_" + str(ROBERTA_MAX_LENGTH)
    print("RoBERTa uses the uploaded checkpoint; no new classifier weights download.")
else:
    MODEL_METADATA = fever.remote_metadata(fever.MODELS[MODEL_FAMILY])
    CONDITION = MODEL_FAMILY + "_3b_nf4_parts"
print("Condition:", CONDITION)
print("Full evaluation target:", len(EVAL_ROWS), "claims. No evaluation has started.")

## 6. Run one engineering check first
Downloads and this one-claim check are enabled at your request. Run sections 1–6 in order; section 5 prints the download estimate before section 6 downloads weights. This check uses **one training-pilot claim**, not a reduced test set. Its article URLs come from that example's retrieved pages; full evaluation uses the exported original top-ten ranking.

Inspect the full prompt, actual article passages and raw output. A successful check verifies execution, not model accuracy. Nothing from this check is pooled into the 19,998-claim result.

In [ ]:
RUN_SMOKE_CHECK = True # @param {type:"boolean"}
if not RUN_SMOKE_CHECK:
    print("Not started. Enable RUN_SMOKE_CHECK after setup.")
elif not ALLOW_MODEL_DOWNLOADS:
    print("Read section 5, enable ALLOW_MODEL_DOWNLOADS, rerun section 4 and section 5, then rerun this cell.")
else:
    from urllib.parse import quote
    import time
    example = fever.read_rows(PROJECT / "data/train_pilot.jsonl")[0]
    pages = list(dict.fromkeys(e["page_id"] for e in example["evidence"]))
    from fetch_wikipedia_articles import page_url
    example = {"id": example["id"], "claim": example["claim"],
               "ranked_articles": [{"rank": i + 1, "url": page_url(p)} for i, p in enumerate(pages)]}
    encoder = fever.ResidentEmbedder(EMBEDDING)
    verifier = None
    try:
        context = fever.context_for(example, PROJECT, RUN_ROOT / "software_check", encoder, EMBEDDING)
        print("Claim:", example["claim"])
        print("Evidence status:", context["status"])
        if context["status"] != "READY":
            raise RuntimeError("Wikipedia preparation failed. Inspect saved error before any full run: " + str(context.get("error")))
        verifier = fever.RobertaVerifier(MODEL_METADATA["path"], ROBERTA_MAX_LENGTH) if MODEL_FAMILY == "roberta" else fever.CausalVerifier(MODEL_METADATA, PROJECT)
        result = verifier.predict(example, context, max_new_tokens=MAX_NEW_TOKENS)
        fever.write_once(RUN_ROOT / "software_check" / (CONDITION + "_" + str(time.time_ns()) + ".json"), result)
        print(json.dumps(result, indent=2, ensure_ascii=False))
        print("Supplied evidence:", json.dumps(context["supplied_evidence"], indent=2, ensure_ascii=False))
    finally:
        if verifier is not None:
            verifier.close()
        del encoder
        fever.release()

## 7. Run / resume the full 19,998-claim evaluation
This is the long operation. It is **off by default**. It attempts the whole official set in its original order and pauses after the session budget. Rerun this cell to resume; do not delete the Drive folder.

Run one model family at a time, then change `MODEL_FAMILY` in section 4 and rerun sections 4–5 and this section. Each family reuses the same saved per-claim RAG passages. Newly fetched full articles and vectors are deleted after each claim; supplied passages, revisions and exact model calls remain.

Failed evidence preparation is an **INVALID execution**, not NEI. Completed invalid predictions remain errors; there is no silent retry. No strict FEVER evidence score is claimed for live article passages. Free Colab may stop early, and GPU availability is not guaranteed.
**Live output:** one progress bar, the current claim and stage, article successes/failures, token counts, prediction, invalid count, GPU memory in the status file, and a short raw-response preview. During a long stage, a heartbeat prints its elapsed time every 20 seconds. A rough ETA starts after three saved claims; it is not a guarantee.

Logs and status are saved in `RUN_ROOT/monitor/CONDITION/`. Repeated infrastructure failures or ten consecutive invalid outputs pause the run for inspection. Saved failures remain errors. These messages do not bypass Colab timeouts.


In [ ]:
RUN_FULL_EVALUATION = False # @param {type:"boolean"}
MAX_NEW_CLAIMS_THIS_SESSION = 0 # @param {type:"integer"}
# 0 means continue until the time budget; a positive number pauses after that many NEW claims.
if not RUN_FULL_EVALUATION:
    print("Full run is off. Target remains all 19,998 claims.")
elif not ALLOW_MODEL_DOWNLOADS:
    print("Model downloads are disabled; inspect section 5 before enabling them.")
else:
    fever.run_condition(EVAL_ROWS, PROJECT, RUN_ROOT, CONDITION, MODEL_FAMILY,
                        MODEL_METADATA, EMBEDDING, minutes=SESSION_MINUTES,
                        max_new_tokens=MAX_NEW_TOKENS, max_length=ROBERTA_MAX_LENGTH,
                        max_new_claims=MAX_NEW_CLAIMS_THIS_SESSION or None)

### Inspect a saved claim or recent events
While the evaluation cell runs, watch its live output. After pausing, stopping or a failure, run this cell to inspect Drive logs and the latest saved prediction. Choose a claim ID for its exact prompt, raw response and supplied passages. Reference labels are read only after a prediction has been saved; they never enter the model prompt.


In [ ]:
INSPECT_CLAIM_ID = "" # @param {type:"string"}
SHOW_FULL_TRACE = False # @param {type:"boolean"}
SHOW_REFERENCE_LABEL = False # @param {type:"boolean"}
fever.monitor_status(RUN_ROOT, CONDITION, last_events=8)
try:
    fever.inspect_saved(RUN_ROOT, CONDITION, claim_id=int(INSPECT_CLAIM_ID) if INSPECT_CLAIM_ID.strip() else None,
        show_reference=SHOW_REFERENCE_LABEL, root=PROJECT, show_full=SHOW_FULL_TRACE)
except FileNotFoundError:
    print("That claim has no saved prediction yet. Check the active stage in the status above.")


## 8. Check progress and score complete conditions
Partial runs show progress only. A final accuracy table is produced only when a condition covers **all 19,998 IDs**. INVALID remains in the denominator. Metrics are label accuracy, macro-F1, per-class precision/recall/F1 and confusion matrices.

A complete official-development result is still a development result. The benchmark has already been inspected. The historical audit found three normalized-text overlaps with the original checkpoint’s training pool; all 19,998 benchmark rows remain in evaluation. This is not an untouched final test.

In [ ]:
conditions = sorted(p.name for p in (RUN_ROOT / "predictions").glob("*") if p.is_dir())
for name in conditions:
    folder = RUN_ROOT / "predictions" / name
    n = sum(p.name != "protocol.json" for p in folder.glob("*.json"))
    print(name, n, "/", len(EVAL_ROWS))
METRICS = fever.score(PROJECT, RUN_ROOT, conditions, require_full=True)
if METRICS:
    import pandas as pd
    display(pd.DataFrame([{"System": name, "Claims": m["n"], "Accuracy (%)": 100*m["accuracy"],
                           "Macro-F1": m["macro_f1"], "Invalid": m["invalid"]} for name, m in METRICS.items()]))
    import matplotlib.pyplot as plt
    for name, m in METRICS.items():
        display(pd.DataFrame([{"Class": label, "Precision (%)": 100*v["precision"],
                              "Recall (%)": 100*v["recall"], "F1 (%)": 100*v["f1"]}
                             for label, v in m["per_class"].items()]))
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.imshow(m["confusion_matrix"], cmap="Blues")
        ax.set_xticks(range(4), ["SUPPORTS", "REFUTES", "NEI", "INVALID"])
        ax.set_yticks(range(3), ["SUPPORTS", "REFUTES", "NEI"])
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(name + " — all 19,998 claims")
        for i, row in enumerate(m["confusion_matrix"]):
            for j, value in enumerate(row): ax.text(j, i, str(value), ha="center", va="center")
        plt.show()

## 9. Optional RoBERTa layer-unfreezing check
This is a **training operation**, off by default. Run it only when you choose to start training.

- `pilot`: included **96 training + 96 validation examples**, used only to verify that training and checkpoints work.
- `full`: requires `train_full.jsonl` and `validation_full.jsonl` from the optional full-training export. The starter bundle intentionally does not prepare this hours-long local retrieval job.
- All conditions start from the **same original task-trained RoBERTa checkpoint**. This is continued fine-tuning, not starting from raw pretrained RoBERTa.
- Freeze all weights, then unfreeze the classification head and the last **1, 2 or 3 Transformer blocks**. Earlier blocks remain frozen.
- Fixed seed 42, 2 epochs, learning rate 2e-5, batch size 4, gradient accumulation 4, maximum 512 tokens. Validation selects the checkpoint; official development labels never train or select it.
- Training uses historical retrieved sentence text for every class, not the new live-article RAG corpus. Report that distribution difference. The pilot includes nonempty NEI evidence to avoid the old empty-input shortcut.

Change only the number of unfrozen layers between controlled conditions. Llama/Qwen here are inference baselines: **this notebook does not implement their fine-tuning**. Quantized LLM adapter training would be a separate experiment and is not equivalent to unfreezing their original last layers.

In [ ]:
RUN_ROBERTA_TRAINING = False # @param {type:"boolean"}
UNFREEZE_LAST_N = 1 # @param [1, 2, 3] {type:"raw"}
TRAIN_SCOPE = "pilot" # @param ["pilot", "full"]
if not RUN_ROBERTA_TRAINING:
    print("Training is off. Pilot training is a software check, not a completed research experiment.")
else:
    final_checkpoint = fever.finetune_roberta(PROJECT, RUN_ROOT,
                    last_layers=UNFREEZE_LAST_N, scope=TRAIN_SCOPE, minutes=SESSION_MINUTES)
    if final_checkpoint:
        print("To evaluate this model, set ROBERTA_CHECKPOINT_OVERRIDE in section 4 to:")
        print(str(final_checkpoint))
        print("Use a distinct condition name for each trained checkpoint; see the next cell.")

### Evaluate a completed fine-tuned checkpoint
Use the full 19,998 inputs again. The condition name includes its layer count and training scope. A pilot-trained condition is explicitly named pilot and must not be called full-training performance.

In [ ]:
RUN_FINETUNED_EVALUATION = False # @param {type:"boolean"}
if not RUN_FINETUNED_EVALUATION:
    print("Fine-tuned evaluation is off.")
else:
    trained = RUN_ROOT / "training" / ("roberta_" + TRAIN_SCOPE + "_last" + str(UNFREEZE_LAST_N))
    if not (trained / "complete.json").is_file():
        raise RuntimeError("Training has not completed. Resume training before evaluating its final checkpoint.")
    name = "roberta_" + TRAIN_SCOPE + "_last" + str(UNFREEZE_LAST_N) + "_rag_" + str(ROBERTA_MAX_LENGTH)
    fever.run_condition(EVAL_ROWS, PROJECT, RUN_ROOT, name, "roberta",
        {"path": str(trained / "final")}, EMBEDDING, minutes=SESSION_MINUTES,
        max_new_tokens=MAX_NEW_TOKENS, max_length=ROBERTA_MAX_LENGTH)

## 10. Download a debug ZIP to send for review
Run this after pausing the evaluation/training cell. The ZIP contains the latest status and events, recent exact prompts/responses with their evidence, condition protocols, available metrics, and training-loss history. It excludes model weights, optimizer files, tokens and caches.

Leave `EXPORT_ALL_PREDICTIONS_CSV` off for a quick debug package. Turn it on after a full run if you also want compact CSVs of every saved prediction; scanning many Drive files can take time. The export does not run inference or change saved results. The ZIP is saved in **FEVER_Project / exports** and offered as a browser download. Upload that downloaded ZIP in our chat.

In [ ]:
EXPORT_DEBUG_ZIP = False # @param {type:"boolean"}
EXPORT_ALL_PREDICTIONS_CSV = False # @param {type:"boolean"}
if not EXPORT_DEBUG_ZIP:
    print("Enable EXPORT_DEBUG_ZIP and run this cell after a session pauses or fails.")
else:
    import csv, io, zipfile, platform
    from collections import deque
    from datetime import datetime, timezone
    from google.colab import files
    export_dir = DRIVE_PROJECT / "exports"
    export_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    export_path = export_dir / ("FEVER_debug_" + stamp + ".zip")
    added = set()
    recent_predictions = []
    def add_file(z, path, archive_name):
        path = Path(path)
        if path.is_file() and archive_name not in added:
            z.write(path, archive_name)
            added.add(archive_name)
    with zipfile.ZipFile(export_path, "x", zipfile.ZIP_DEFLATED) as z:
        z.writestr("export_info.json", json.dumps({"created_utc": stamp, "experiment": EXPERIMENT_NAME,
            "condition_selected": CONDITION, "python": platform.python_version(),
            "scope": "debug subset; recent full traces, not every model response",
            "all_predictions_csv_requested": EXPORT_ALL_PREDICTIONS_CSV}, indent=2))
        add_file(z, PROJECT / "manifest.json", "project_manifest.json")
        for status_path in (RUN_ROOT / "monitor").glob("*/status.json"):
            name = status_path.parent.name
            add_file(z, status_path, "monitor/" + name + "/status.json")
            logs = sorted(status_path.parent.glob("session_*.jsonl"))
            if logs:
                with logs[-1].open() as stream:
                    tail = list(deque(stream, maxlen=2000))
                z.writestr("monitor/" + name + "/recent_events.jsonl", "".join(tail))
                for line in reversed(tail):
                    event = json.loads(line)
                    if event.get("event") == "claim_saved" and event.get("claim_id") is not None:
                        key = (name, str(event["claim_id"]))
                        if key not in recent_predictions:
                            recent_predictions.append(key)
                            if sum(n == name for n, _ in recent_predictions) >= 10:
                                break
        for condition_dir in (RUN_ROOT / "predictions").glob("*"):
            if not condition_dir.is_dir():
                continue
            add_file(z, condition_dir / "protocol.json", "conditions/" + condition_dir.name + "/protocol.json")
            if EXPORT_ALL_PREDICTIONS_CSV:
                print("Exporting compact predictions:", condition_dir.name, flush=True)
                buffer = io.StringIO()
                columns = ["id", "claim", "label", "model_label", "evidence_status", "input_tokens", "output_tokens", "wall_seconds", "error"]
                writer = csv.DictWriter(buffer, fieldnames=columns, extrasaction="ignore")
                writer.writeheader()
                for path in condition_dir.glob("*.json"):
                    if path.name == "protocol.json":
                        continue
                    writer.writerow(json.loads(path.read_text()))
                z.writestr("conditions/" + condition_dir.name + "/predictions.csv", buffer.getvalue())
        for name, claim_id in recent_predictions:
            add_file(z, RUN_ROOT / "predictions" / name / (claim_id + ".json"), "traces/" + name + "/" + claim_id + ".json")
            add_file(z, RUN_ROOT / "evidence" / ("claim_" + claim_id + ".json"), "evidence/" + claim_id + ".json")
        for path in RUN_ROOT.glob("metrics_*.json"):
            add_file(z, path, "metrics/" + path.name)
        for pattern in ["training/*/protocol.json", "training/*/complete.json",
                        "training/*/monitor/status.json", "training/*/checkpoints/checkpoint-*/trainer_state.json"]:
            for path in RUN_ROOT.glob(pattern):
                add_file(z, path, str(path.relative_to(RUN_ROOT)))
        for path in RUN_ROOT.glob("training/*/monitor/session_*.jsonl"):
            with path.open() as stream:
                tail = list(deque(stream, maxlen=300))
            z.writestr(str(path.relative_to(RUN_ROOT)), "".join(tail))
        # Include the small engineering check too, if no long evaluation has run yet.
        for path in (RUN_ROOT / "software_check").glob("*.json"):
            add_file(z, path, "software_check/" + path.name)
        for path in (RUN_ROOT / "software_check" / "evidence").glob("*.json"):
            add_file(z, path, "software_check/evidence/" + path.name)
    print("Saved debug ZIP:", export_path)
    print("Size:", round(export_path.stat().st_size / 2**20, 2), "MiB")
    print("If your browser blocks the download, get this ZIP from FEVER_Project / exports in Drive.")
    files.download(str(export_path))

## What this notebook does not claim
- No GPU run has been executed by the notebook author on your Colab account. Local tests cover data guards, response validation and layer selection, not cloud throughput or final accuracy.
- The 3B NF4 models and HF decoder differ from the earlier 8B Ollama experiment. New results need their own names and must not replace the old results.
- The three model families have different architectures and effective context budgets. Same passages do not imply identical retained evidence.
- This does not implement a new symbolic architecture or a Reddit study. Your professor's intended Reddit task still needs clarification.
- Free Colab is time-limited. Save Drive outputs, resume batches, and measure throughput before estimating a full-run completion date. Do not use keep-alive tricks to bypass service limits.

All source code used by a condition is in the uploaded bundle under `src/`, with SHA-256 hashes in `manifest.json`. Exact prompts, responses, retained inputs and evidence revisions are saved under your Drive run folder.